# 12.2 规划 (Planning)

规划使Agent能将复杂任务分解为可执行的步骤序列，是高级推理的核心能力。

本节涵盖：
- 任务分解与CoT/ReWOO
- Tree of Thought与MCTS搜索
- 反思与自我修正 (Reflexion/Self-Refine)
- LATS语言Agent树搜索
- 产业级规划系统设计

## 1. 任务分解策略

### Chain of Thought (CoT)
逐步推理，每步输出中间结果。简单但无法回溯。

### ReWOO (Reasoning WithOut Observation)
先规划所有步骤（不执行），再依次执行。优势：
- 减少上下文中的工具调用结果，节省token
- 规划与执行解耦，便于并行执行
- 避免工具调用结果对后续规划的偏见

### Plan-and-Solve
两阶段：先生成完整计划，再逐步执行。适合结构化任务。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class TaskDecomposer(nn.Module):
    def __init__(self, d_model=128, max_subtasks=5, d_vocab=64):
        super().__init__()
        self.d_model = d_model
        self.max_subtasks = max_subtasks
        self.task_encoder = nn.TransformerEncoderLayer(d_model, nhead=4, dim_feedforward=256, batch_first=True)
        self.subtask_decoder = nn.GRU(d_model, 64, num_layers=2, batch_first=True)
        self.subtask_output = nn.Linear(64, d_vocab)
        self.stop_head = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
        self.dependency_head = nn.Sequential(nn.Linear(64 * 2, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def decompose(self, task_embed):
        encoded = self.task_encoder(task_embed.unsqueeze(1)).squeeze(1)
        subtasks = []
        hidden = torch.zeros(2, 1, 64, device=task_embed.device)
        current = encoded
        for i in range(self.max_subtasks):
            out, hidden = self.subtask_decoder(current.unsqueeze(1), hidden)
            subtask_repr = out[:, -1, :]
            stop_prob = self.stop_head(subtask_repr)
            subtasks.append({'repr': subtask_repr, 'stop_prob': stop_prob.item()})
            if stop_prob.item() > 0.7:
                break
            current = subtask_repr
        return subtasks

    def build_dependency_graph(self, subtasks):
        n = len(subtasks)
        dep_matrix = torch.zeros(n, n)
        for i in range(n):
            for j in range(i):
                pair = torch.cat([subtasks[i]['repr'], subtasks[j]['repr']], dim=-1)
                dep = self.dependency_head(pair)
                dep_matrix[i, j] = dep.item()
        return dep_matrix

class ReWOOPlanner(nn.Module):
    def __init__(self, d_model=128, n_tools=3, max_steps=5):
        super().__init__()
        self.d_model = d_model
        self.n_tools = n_tools
        self.max_steps = max_steps
        self.plan_encoder = nn.GRU(d_model, 64, num_layers=2, batch_first=True)
        self.tool_head = nn.Linear(64, n_tools)
        self.arg_head = nn.Linear(64, d_model)
        self.finish_head = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())

    def plan(self, task_embed):
        plans = []
        hidden = torch.zeros(2, 1, 64, device=task_embed.device)
        current = task_embed
        for step in range(self.max_steps):
            out, hidden = self.plan_encoder(current.unsqueeze(1), hidden)
            plan_state = out[:, -1, :]
            tool_logits = self.tool_head(plan_state)
            arg = self.arg_head(plan_state)
            finish_prob = self.finish_head(plan_state)
            plans.append({
                'tool_logits': tool_logits,
                'arg': arg,
                'finish_prob': finish_prob.item(),
                'state': plan_state
            })
            if finish_prob.item() > 0.7:
                break
            current = arg
        return plans

decomposer = TaskDecomposer(d_model=128)
rewwoo = ReWOOPlanner(d_model=128, n_tools=3)

task = torch.randn(1, 128)

print('=== Task Decomposition ===')
subtasks = decomposer.decompose(task)
print(f'Task decomposed into {len(subtasks)} subtasks:')
for i, st in enumerate(subtasks):
    print(f'  Subtask {i}: stop_prob={st["stop_prob"]:.3f}')

dep_matrix = decomposer.build_dependency_graph(subtasks)
print(f'\nDependency matrix:')
for i in range(dep_matrix.size(0)):
    deps = [j for j in range(dep_matrix.size(1)) if dep_matrix[i, j] > 0.5]
    print(f'  Subtask {i} depends on: {deps if deps else "nothing"}')

print(f'\n=== ReWOO Planning (Plan First, Execute Later) ===')
plans = rewwoo.plan(task)
print(f'Generated {len(plans)} planned steps:')
for i, p in enumerate(plans):
    tool = p['tool_logits'].argmax(dim=-1).item()
    print(f'  Step {i}: tool={tool}, finish_prob={p["finish_prob"]:.3f}')

print(f'\nKey: ReWOO decouples planning from execution, reducing token usage and bias.')
print(f'Dependency graphs enable parallel execution of independent subtasks.')

## 2. Tree of Thought (ToT)

**核心思想**：将推理过程视为搜索树，每个节点是一个推理状态，每条边是一个推理步骤。

**与CoT的区别**：
- CoT是线性链，无法回溯
- ToT是树结构，可以探索多条路径，回溯错误方向

**关键组件**：
1. **状态生成器**：从当前状态生成多个候选下一步
2. **价值评估器**：评估每个候选状态的价值
3. **搜索策略**：BFS（广度优先）或DFS（深度优先）

**产业应用**：游戏AI、数学推理、代码生成中的多路径探索。

In [ ]:
class TreeOfThought(nn.Module):
    def __init__(self, d_model=128, n_branches=3, max_depth=4):
        super().__init__()
        self.n_branches = n_branches
        self.max_depth = max_depth
        self.state_generator = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.value_net = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, 1), nn.Sigmoid()
        )
        self.noise_scale = nn.Parameter(torch.tensor(0.3))

    def generate_candidates(self, state):
        candidates = []
        for i in range(self.n_branches):
            noise = torch.randn_like(state) * self.noise_scale
            candidate = self.state_generator(state + noise)
            value = self.value_net(candidate)
            candidates.append((candidate, value.item()))
        return candidates

    def bfs_search(self, initial_state, beam_width=2):
        current_beam = [(initial_state, 0.0, [])]
        all_paths = []
        for depth in range(self.max_depth):
            next_beam = []
            for state, cum_value, path in current_beam:
                candidates = self.generate_candidates(state)
                for cand_state, cand_value in candidates:
                    new_path = path + [(depth, cand_value)]
                    next_beam.append((cand_state, cum_value + cand_value, new_path))
            next_beam.sort(key=lambda x: x[1], reverse=True)
            current_beam = next_beam[:beam_width]
            all_paths = current_beam
        best_state, best_value, best_path = current_beam[0]
        return best_path, best_value

    def dfs_search(self, initial_state, value_threshold=0.3):
        best_path = []
        best_value = 0
        current = initial_state
        for depth in range(self.max_depth):
            candidates = self.generate_candidates(current)
            candidates.sort(key=lambda x: x[1], reverse=True)
            found = False
            for cand_state, cand_value in candidates:
                if cand_value >= value_threshold:
                    best_path.append((depth, cand_value))
                    current = cand_state
                    best_value = cand_value
                    found = True
                    break
            if not found:
                best_path.append((depth, candidates[0][1]))
                current = candidates[0][0]
                best_value = candidates[0][1]
        return best_path, best_value

tot = TreeOfThought(d_model=128, n_branches=3, max_depth=4)
initial = torch.randn(1, 128)

print('=== Tree of Thought ===')
print(f'Config: branches={tot.n_branches}, max_depth={tot.max_depth}')

bfs_path, bfs_value = tot.bfs_search(initial, beam_width=2)
print(f'\nBFS Search (beam_width=2):')
for depth, value in bfs_path:
    print(f'  Depth {depth}: value={value:.4f}')
print(f'Total path value: {bfs_value:.4f}')

dfs_path, dfs_value = tot.dfs_search(initial, value_threshold=0.3)
print(f'\nDFS Search (threshold=0.3):')
for depth, value in dfs_path:
    print(f'  Depth {depth}: value={value:.4f}')
print(f'Final value: {dfs_value:.4f}')

print(f'\nKey: BFS explores broadly (good for wide solution spaces),')
print(f'DFS explores deeply (good for sequential reasoning chains).')

## 3. 蒙特卡洛树搜索 (MCTS)

MCTS是AlphaGo/AlphaZero的核心算法，平衡探索与利用：

### 四个阶段
1. **选择 (Selection)**：用UCB公式选择最有潜力的节点
2. **扩展 (Expansion)**：为选中节点生成子节点
3. **模拟 (Rollout)**：从新节点快速模拟到终局
4. **回传 (Backpropagation)**：将结果向上传播更新节点统计

### UCB1公式
$$UCB1 = \bar{X}_j + c\sqrt{\frac{\ln N}{n_j}}$$
- $\bar{X}_j$：节点j的平均价值
- $N$：父节点访问次数
- $n_j$：节点j访问次数
- $c$：探索常数

**产业应用**：AlphaGo、AlphaCode、推理增强LLM。

In [ ]:
class MCTSNode:
    def __init__(self, state, parent=None, action=None):
        self.state = state
        self.parent = parent
        self.action = action
        self.children = []
        self.visits = 0
        self.total_value = 0.0
        self.prior = 1.0

    @property
    def q_value(self):
        return self.total_value / max(self.visits, 1)

    def ucb1(self, c=1.414):
        if self.visits == 0:
            return float('inf')
        exploitation = self.q_value
        exploration = c * math.sqrt(math.log(max(self.parent.visits, 1)) / max(self.visits, 1))
        return exploitation + exploration

    def puct(self, c_puct=1.5):
        if self.visits == 0:
            return self.prior * math.sqrt(max(self.parent.visits, 1))
        q = self.q_value
        u = c_puct * self.prior * math.sqrt(max(self.parent.visits, 1)) / (1 + self.visits)
        return q + u

class MCTS:
    def __init__(self, d_model=128, n_actions=4, n_simulations=50):
        self.d_model = d_model
        self.n_actions = n_actions
        self.n_simulations = n_simulations
        self.policy_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, n_actions), nn.Softmax(dim=-1)
        )
        self.value_net = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Tanh()
        )
        self.transition_net = nn.Sequential(
            nn.Linear(d_model + 1, 64), nn.ReLU(), nn.Linear(64, d_model)
        )

    def select(self, node):
        while node.children:
            node = max(node.children, key=lambda c: c.puct())
        return node

    def expand(self, node):
        action_probs = self.policy_net(node.state)
        for action in range(self.n_actions):
            action_input = torch.cat([node.state, torch.tensor([[action]], dtype=torch.float)], dim=-1)
            next_state = self.transition_net(action_input)
            child = MCTSNode(next_state, parent=node, action=action)
            child.prior = action_probs[0, action].item()
            node.children.append(child)

    def evaluate(self, node):
        return self.value_net(node.state).item()

    def backpropagate(self, node, value):
        while node is not None:
            node.visits += 1
            node.total_value += value
            value = -value
            node = node.parent

    def search(self, initial_state):
        root = MCTSNode(initial_state)
        for _ in range(self.n_simulations):
            leaf = self.select(root)
            if leaf.visits > 0:
                self.expand(leaf)
                if leaf.children:
                    leaf = leaf.children[0]
            value = self.evaluate(leaf)
            self.backpropagate(leaf, value)
        best_child = max(root.children, key=lambda c: c.visits) if root.children else root
        return best_child.action, best_child.q_value, root.visits

mcts = MCTS(d_model=128, n_actions=4, n_simulations=30)
initial = torch.randn(1, 128)

print('=== Monte Carlo Tree Search ===')
print(f'Simulations: {mcts.n_simulations}')

best_action, best_value, total_visits = mcts.search(initial)
print(f'\nSearch result:')
print(f'  Best action: {best_action}')
print(f'  Best value: {best_value:.4f}')
print(f'  Total visits: {total_visits}')

print(f'\nMulti-step planning:')
state = initial
for step in range(3):
    action, value, visits = mcts.search(state)
    action_input = torch.cat([state, torch.tensor([[action]], dtype=torch.float)], dim=-1)
    state = mcts.transition_net(action_input)
    print(f'  Step {step+1}: action={action}, value={value:.4f}, visits={visits}')

print(f'\nKey: MCTS balances exploration (trying new paths) and exploitation (using known good paths).')
print(f'PUCT formula (from AlphaZero) uses prior policy to guide search more efficiently than UCB1.')

## 4. 反思与自我修正

### Reflexion
Agent执行任务后进行自我反思，将反思结果作为下次尝试的额外输入，迭代改进。
- 不更新模型权重，而是更新上下文
- 反思内容包含：失败原因分析、改进策略
- 通常3-5次迭代即可收敛

### Self-Refine
模型对自身输出进行多轮自我批评和修正：
1. 生成初始输出
2. 自我批评：找出输出中的问题
3. 自我修正：基于批评改进输出
4. 重复2-3直到满意

### 产业实践
- Reflexion在HumanEval上比标准CoT提升约20%
- Self-Refine在写作任务上显著提升质量
- 两者可组合使用：Reflexion做宏观策略调整，Self-Refine做微观输出优化

In [ ]:
class ReflexionAgent(nn.Module):
    def __init__(self, d_model=128, n_actions=4, max_trials=5):
        super().__init__()
        self.d_model = d_model
        self.n_actions = n_actions
        self.max_trials = max_trials
        self.action_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, n_actions)
        )
        self.reflect_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(),
            nn.Linear(64, d_model)
        )
        self.evaluate_net = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )
        self.feedback_encoder = nn.Linear(d_model, d_model)

    def act(self, state):
        logits = self.action_net(state)
        return logits

    def reflect(self, state, feedback):
        combined = torch.cat([state, self.feedback_encoder(feedback)], dim=-1)
        return self.reflect_net(combined)

    def evaluate(self, state):
        return self.evaluate_net(state)

    def run_with_reflexion(self, initial_state, success_threshold=0.8):
        state = initial_state
        reflections = []
        trajectory = []
        for trial in range(self.max_trials):
            action_logits = self.act(state)
            action = action_logits.argmax(dim=-1).item()
            score = self.evaluate(state).item()
            trajectory.append({'trial': trial, 'action': action, 'score': score})
            if score >= success_threshold:
                return trajectory, True
            feedback = torch.randn(1, self.d_model)
            reflection = self.reflect(state, feedback)
            reflections.append(reflection)
            state = state + 0.1 * reflection
            for prev_r in reflections[:-1]:
                state = state + 0.05 * prev_r
        return trajectory, False

class SelfRefineAgent(nn.Module):
    def __init__(self, d_model=128, max_rounds=3):
        super().__init__()
        self.d_model = d_model
        self.max_rounds = max_rounds
        self.generate_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.critic_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.refine_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.quality_net = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def forward(self, task_embed):
        output = self.generate_net(task_embed)
        history = [output.clone()]
        for round_num in range(self.max_rounds):
            critic_input = torch.cat([task_embed, output], dim=-1)
            criticism = self.critic_net(critic_input)
            refine_input = torch.cat([output, criticism], dim=-1)
            output = self.refine_net(refine_input)
            quality = self.quality_net(output).item()
            history.append(output.clone())
            if quality > 0.85:
                break
        return output, history

reflex_agent = ReflexionAgent(d_model=128)
refine_agent = SelfRefineAgent(d_model=128)

state = torch.randn(1, 128)
task = torch.randn(1, 128)

print('=== Reflexion ===')
trajectory, success = reflex_agent.run_with_reflexion(state, success_threshold=0.7)
for t in trajectory:
    print(f'  Trial {t["trial"]}: action={t["action"]}, score={t["score"]:.4f}')
print(f'Result: {"Success" if success else "Max trials reached"}')

print(f'\n=== Self-Refine ===')
final_output, history = refine_agent(task)
for i, h in enumerate(history):
    quality = refine_agent.quality_net(h).item()
    label = 'Initial' if i == 0 else f'Refine {i}'
    print(f'  {label}: quality={quality:.4f}, norm={h.norm():.3f}')

print(f'\nKey: Reflexion updates the state/context between trials (macro-level).')
print(f'Self-Refine iteratively improves a single output (micro-level).')
print(f'Both are training-free — they improve through structured self-feedback.')

## 5. LATS — 语言Agent树搜索

**LATS** (Language Agent Tree Search) 结合了MCTS和LLM推理：
- 用LLM生成候选行动（替代传统MCTS的均匀展开）
- 用LLM评估状态价值（替代传统价值网络）
- 用LLM反思失败路径（新增组件）

**与标准MCTS的区别**：
| 组件 | 标准MCTS | LATS |
|------|---------|------|
| 展开 | 均匀/随机 | LLM生成 |
| 评估 | 价值网络/模拟 | LLM评估 |
| 反思 | 无 | LLM反思 |
| 搜索空间 | 离散动作 | 自然语言推理步骤 |

**优势**：在复杂推理任务上显著优于CoT和ToT，因为MCTS提供了系统性搜索框架。

In [ ]:
class LATSNode(MCTSNode):
    def __init__(self, state, parent=None, action=None, reflection=None):
        super().__init__(state, parent, action)
        self.reflection = reflection
        self.is_terminal = False

class LATS:
    def __init__(self, d_model=128, n_actions=4, n_simulations=30, reflection_weight=0.3):
        self.d_model = d_model
        self.n_actions = n_actions
        self.n_simulations = n_simulations
        self.reflection_weight = reflection_weight
        self.policy_net = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, n_actions), nn.Softmax(dim=-1)
        )
        self.value_net = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Tanh()
        )
        self.transition_net = nn.Sequential(
            nn.Linear(d_model + 1, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.reflection_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.terminal_net = nn.Sequential(
            nn.Linear(d_model, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )

    def select(self, node):
        while node.children:
            node = max(node.children, key=lambda c: c.puct())
        return node

    def expand_with_reflection(self, node):
        action_probs = self.policy_net(node.state)
        reflection_state = node.state
        if node.reflection is not None:
            reflection_state = node.state + self.reflection_weight * node.reflection
        for action in range(self.n_actions):
            action_input = torch.cat([reflection_state, torch.tensor([[action]], dtype=torch.float)], dim=-1)
            next_state = self.transition_net(action_input)
            child = LATSNode(next_state, parent=node, action=action)
            child.prior = action_probs[0, action].item()
            child.is_terminal = self.terminal_net(next_state).item() > 0.8
            node.children.append(child)

    def evaluate(self, node):
        return self.value_net(node.state).item()

    def backpropagate(self, node, value):
        while node is not None:
            node.visits += 1
            node.total_value += value
            value = -value
            node = node.parent

    def reflect_on_failure(self, node):
        if node.parent is not None and node.parent.reflection is not None:
            reflection = self.reflection_net(torch.cat([node.state, node.parent.reflection], dim=-1))
        else:
            reflection = self.reflection_net(torch.cat([node.state, torch.zeros_like(node.state)], dim=-1))
        return reflection

    def search(self, initial_state):
        root = LATSNode(initial_state)
        best_terminal = None
        best_terminal_value = float('-inf')
        for sim in range(self.n_simulations):
            leaf = self.select(root)
            if leaf.is_terminal:
                value = self.evaluate(leaf)
                if value > best_terminal_value:
                    best_terminal = leaf
                    best_terminal_value = value
            else:
                if leaf.visits > 0:
                    self.expand_with_reflection(leaf)
                    if leaf.children:
                        leaf = leaf.children[0]
                value = self.evaluate(leaf)
                if value < -0.5:
                    reflection = self.reflect_on_failure(leaf)
                    leaf.reflection = reflection
            self.backpropagate(leaf, value)
        best_child = max(root.children, key=lambda c: c.visits) if root.children else root
        return best_child.action, best_child.q_value, root.visits, best_terminal

lats = LATS(d_model=128, n_actions=4, n_simulations=20)
initial = torch.randn(1, 128)

print('=== LATS: Language Agent Tree Search ===')
action, value, visits, terminal = lats.search(initial)
print(f'Best action: {action}, value: {value:.4f}, total visits: {visits}')
if terminal:
    print(f'Terminal node found with value: {lats.evaluate(terminal):.4f}')

print(f'\nMulti-step LATS planning:')
state = initial
for step in range(3):
    action, value, visits, terminal = lats.search(state)
    action_input = torch.cat([state, torch.tensor([[action]], dtype=torch.float)], dim=-1)
    state = lats.transition_net(action_input)
    print(f'  Step {step+1}: action={action}, value={value:.4f}')

print(f'\nKey: LATS adds reflection to MCTS — failed paths generate insights that guide future search.')
print(f'Terminal detection allows early stopping when a good solution is found.')

## 6. 产业级规划系统设计

### 规划系统的工程挑战

1. **延迟控制**：MCTS/LATS的搜索过程可能很慢，需要设置超时和预算
2. **计划缓存**：相似任务的计划可以复用，避免重复搜索
3. **计划验证**：执行前验证计划的可行性（类型检查、依赖检查）
4. **自适应深度**：简单任务浅搜索，复杂任务深搜索
5. **人机协作**：关键决策点请求人类确认

### 规划模式选择指南

| 任务复杂度 | 推荐方法 | 典型场景 |
|-----------|---------|----------|
| 简单(1-2步) | 直接CoT | 简单问答 |
| 中等(3-5步) | ReAct/ReWOO | 多工具调用 |
| 复杂(多路径) | ToT/BFS | 创意生成 |
| 高复杂度 | MCTS/LATS | 数学推理、代码生成 |
| 需要迭代改进 | Reflexion/Self-Refine | 写作、优化 |

### 生产环境最佳实践
- 设置搜索预算（最大模拟次数、最大token数）
- 实现计划检查点，允许中断和恢复
- 记录搜索轨迹，便于调试和优化
- 对高频任务建立计划模板库

In [ ]:
class AdaptivePlanner:
    def __init__(self, d_model=128, n_tools=3):
        self.d_model = d_model
        self.n_tools = n_tools
        self.complexity_estimator = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 3), nn.Softmax(dim=-1)
        )
        self.cot_planner = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, n_tools))
        self.react_planner = ReActAgent(d_model, n_tools, max_steps=4)
        self.tot_planner = TreeOfThought(d_model, n_branches=3, max_depth=3)
        self.mcts_planner = MCTS(d_model, n_actions=n_tools + 1, n_simulations=20)
        self.plan_cache = {}
        self.stats = {'cot': 0, 'react': 0, 'tot': 0, 'mcts': 0, 'cache_hits': 0}

    def plan(self, task_embed, use_cache=True):
        cache_key = tuple((task_embed * 100).int().flatten().tolist()[:8])
        if use_cache and cache_key in self.plan_cache:
            self.stats['cache_hits'] += 1
            return self.plan_cache[cache_key]

        complexity = self.complexity_estimator(task_embed)
        level = complexity.argmax(dim=-1).item()

        if level == 0:
            result = self._plan_cot(task_embed)
            self.stats['cot'] += 1
        elif level == 1:
            result = self._plan_react(task_embed)
            self.stats['react'] += 1
        elif level == 2:
            result = self._plan_mcts(task_embed)
            self.stats['mcts'] += 1
        else:
            result = self._plan_tot(task_embed)
            self.stats['tot'] += 1

        self.plan_cache[cache_key] = result
        return result

    def _plan_cot(self, task_embed):
        logits = self.cot_planner(task_embed)
        return {'method': 'CoT', 'action': logits.argmax(dim=-1).item(), 'confidence': logits.max().item()}

    def _plan_react(self, task_embed):
        action_logits, should_finish, answer, thought = self.react_planner(task_embed)
        return {'method': 'ReAct', 'action': action_logits.argmax(dim=-1).item(),
                'finish_prob': should_finish.item()}

    def _plan_tot(self, task_embed):
        path, value = self.tot_planner.bfs_search(task_embed, beam_width=2)
        return {'method': 'ToT', 'path_length': len(path), 'value': value}

    def _plan_mcts(self, task_embed):
        action, value, visits, _ = self.mcts_planner.search(task_embed)
        return {'method': 'MCTS', 'action': action, 'value': value, 'visits': visits}

planner = AdaptivePlanner(d_model=128, n_tools=3)

print('=== Adaptive Planning System ===')
tasks = [
    ('Simple query', torch.randn(1, 128)),
    ('Multi-step task', torch.randn(1, 128)),
    ('Complex reasoning', torch.randn(1, 128)),
    ('Cached simple query', torch.randn(1, 128)),
]

for name, task in tasks:
    result = planner.plan(task, use_cache=True)
    print(f'\n{name}:')
    print(f'  Method: {result["method"]}')
    for k, v in result.items():
        if k != 'method':
            print(f'  {k}: {v}')

print(f'\nPlanner stats: {planner.stats}')
print(f'\nKey: Adaptive planning selects the right algorithm based on task complexity.')
print(f'Plan caching avoids redundant computation for similar tasks.')
print(f'In production, complexity estimation can be a lightweight classifier or rule-based.')

## 7. 规划总结

| 方法 | 搜索空间 | 回溯能力 | 计算成本 | 适用场景 |
|------|---------|---------|---------|----------|
| CoT | 线性链 | 无 | 低 | 简单推理 |
| ReAct | 线性+工具 | 有限(重试) | 中 | 工具调用 |
| ReWOO | 先规划后执行 | 无 | 中 | 结构化任务 |
| ToT | 树 | 有 | 中-高 | 多路径探索 |
| MCTS | 树+模拟 | 有 | 高 | 复杂决策 |
| LATS | 树+反思 | 有 | 高 | 复杂推理 |
| Reflexion | 线性+反思 | 有限 | 中 | 迭代改进 |
| Self-Refine | 线性+修正 | 有限 | 低-中 | 输出优化 |

**前沿方向**：
- **Q***：结合过程奖励模型(PRM)的搜索
- **AlphaCode 2**：竞赛级代码生成的搜索策略
- **rStar**：自博弈推理搜索
- **Think-on-Graph**：知识图谱增强的推理搜索